# 07 – LG vs rivals: tests clustered by product
Run from inside `findings/`. Recomputes the LG-vs-rivals numbers in Sections 4.2, 5.3 and 7.4 of the report.

1. Share of 1–2 star reviews mentioning each issue, LG vs rivals, tested with a logistic regression **clustered by product** (several reviews come from the same product, so they are not independent). The plain two-proportion z-test is shown next to it for comparison.
2. Support-mention rates across all reviews and the impact estimate in stars.
3. Recall of the support tag for LG vs rivals, against the AI labels in `validation/`.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest

d = pd.read_csv("../data_clean/reviews_laundry_features.csv")
tags = [c for c in d.columns if c.startswith("a_")]
low = d[d["low"] == 1].copy()
print(f"1-2 star reviews: LG {int(low.is_lg.sum()):,}, rivals {int((low.is_lg == 0).sum()):,}")

1-2 star reviews: LG 609, rivals 1,838


## 1. LG vs rivals among 1–2 star reviews
For each issue: `mention ~ is_lg` on 1–2 star reviews, standard errors clustered by product. The p-value on `is_lg` tests whether LG's share differs from rivals'.

In [2]:
groups = pd.factorize(low["product_id"])[0]
rows = []
for t in tags:
    fit = smf.logit(f"{t} ~ is_lg", data=low).fit(disp=0, cov_type="cluster", cov_kwds={"groups": groups})
    lg, rv = low[low.is_lg == 1][t], low[low.is_lg == 0][t]
    z_p = proportions_ztest([lg.sum(), rv.sum()], [len(lg), len(rv)])[1]
    rows.append({"issue": t[2:], "LG %": 100 * lg.mean(), "rivals %": 100 * rv.mean(),
                 "gap (pts)": 100 * (lg.mean() - rv.mean()),
                 "z-test p": z_p, "clustered p": fit.pvalues["is_lg"]})
lg_vs_rivals = pd.DataFrame(rows).sort_values("gap (pts)", ascending=False).round(3)
lg_vs_rivals["significant (clustered, 5%)"] = lg_vs_rivals["clustered p"] < 0.05
lg_vs_rivals.to_csv("lg_vs_rivals_clustered.csv", index=False)
lg_vs_rivals

,issue,LG %,rivals %,gap (pts),z-test p,clustered p,"significant (clustered, 5%)"
7,support,22.332,12.840,9.492,0.000,0.000,True
5,drying_clothes,12.808,8.923,3.885,0.005,0.115,False
9,return_refund,8.867,6.964,1.903,0.121,0.142,False
4,value,30.542,30.359,0.183,0.932,0.930,False
2,cleaning,18.062,18.226,-0.164,0.928,0.951,False
0,reliability,34.647,34.875,-0.228,0.919,0.928,False
8,delivery_install,0.985,1.251,-0.266,0.599,0.600,False
6,repair_service,48.440,49.075,-0.635,0.786,0.828,False
3,design_usability,19.704,21.273,-1.569,0.409,0.465,False
1,noise,10.509,15.452,-4.943,0.002,0.016,True


**Reading:** support (22.3% vs 12.8%) and noise (10.5% vs 15.5%) stay significant with clustering. The drying gap (12.8% vs 8.9%) is significant with the plain z-test but **not** once reviews are clustered by product, because it comes from a few models with many reviews.

## 2. Impact estimate (Section 7.4)
OLS of the 1–5 star rating on all tags with brand, year and verified-purchase controls (same as notebook 03), clustered by product, gives the stars associated with a support mention. Multiplying it by the LG-vs-rivals gap in support mentions **across all reviews** gives the associated difference in LG's average rating. (The coefficient is estimated on all reviews, so the gap must be measured on all reviews too.)

In [3]:
d["verified"] = d["verified_purchase"].astype(int)
# same specification as the OLS robustness check in notebook 03 (section 4.4)
ols = smf.ols("review_rating ~ " + " + ".join(tags) + " + C(brand, Treatment('LG')) + verified + C(year)", data=d).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(d["product_id"])[0]})
b = ols.params["a_support"]
lg_rate, rv_rate = d[d.is_lg == 1].a_support.mean(), d[d.is_lg == 0].a_support.mean()
print(f"Support coefficient (stars): {b:.2f}")
print(f"Support mentions, all reviews: LG {lg_rate:.1%}, rivals {rv_rate:.1%}, gap {lg_rate - rv_rate:.1%}")
print(f"Associated difference in LG's average rating: {(lg_rate - rv_rate) * -b:+.3f} stars")

Support coefficient (stars): -0.57
Support mentions, all reviews: LG 12.4%, rivals 9.8%, gap 2.6%
Associated difference in LG's average rating: +0.015 stars


## 3. Support-tag recall, LG vs rivals
Against the AI labels on 200 random reviews: of the reviews the AI coder labels as a support complaint, the share the keyword tag catches.

In [4]:
lab = pd.read_csv("../validation/ai_labels_200.csv")
lab["kw_support"] = d.set_index("review_id").loc[lab.review_id, "a_support"].values
for name, g in lab.groupby(lab.is_lg.map({1: "LG", 0: "rivals"})):
    hits = g[g.ai_support == 1]
    print(f"{name}: AI support complaints {len(hits)}, caught by keywords {int(hits.kw_support.sum())}, recall {hits.kw_support.mean():.0%}")

LG: AI support complaints 13, caught by keywords 5, recall 38%
rivals: AI support complaints 26, caught by keywords 11, recall 42%
